Setup: load and clean via src/preprocessing

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.preprocessing import load_and_clean, FEATURE_COLS

df = load_and_clean()
print(df.shape)
print(df["label"].value_counts())
print("distinct groups:", df["group_key"].nunique())

(5899, 21)
label
Category_1    5215
Category_2     631
Category_3      23
Category_4      16
Category_6      12
Category_5       2
Name: count, dtype: int64
distinct groups: 5219


Setting Category_5 aside, then a group-aware stratified holdout

In [2]:
from sklearn.model_selection import StratifiedGroupKFold

# step 1: pull the 2 Category_5 rows aside
cat5 = df[df["label"] == "Category_5"]
rest = df[df["label"] != "Category_5"].reset_index(drop=True)
print("Category_5 held aside:", len(cat5), "| remaining:", len(rest))

# step 2: group-aware stratified split, one fold of 10 becomes the holdout
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
train_idx, hold_idx = next(sgkf.split(rest, rest["label"], groups=rest["group_key"]))

train = rest.iloc[train_idx]
holdout = rest.iloc[hold_idx]

print(f"\nholdout: {len(holdout)} rows ({len(holdout)/len(rest):.1%})")
print("\nclass counts:")
print(pd.DataFrame({
    "train": train["label"].value_counts(),
    "holdout": holdout["label"].value_counts(),
}).fillna(0).astype(int))

Category_5 held aside: 2 | remaining: 5897

holdout: 590 rows (10.0%)

class counts:
            train  holdout
label                     
Category_1   4693      522
Category_2    568       63
Category_3     21        2
Category_4     14        2
Category_6     11        1


Verifying the split: no group straddles it, and the signal tokens stay in training

In [3]:
# no group may straddle the split
overlap = set(train["group_key"]) & set(holdout["group_key"])
print("groups on both sides:", len(overlap))

# rare-class signal tokens must be present in training
checks = {
    "Col4": ["Word575", "Word868", "Word905"],
    "Col6": ["Word576", "Word571"],
    "Col1": ["Word19", "Word85", "Word101"],
}
for col, toks in checks.items():
    for t in toks:
        pat = rf"\b{t}\b"
        n_tr = train[col].str.contains(pat, regex=True).sum()
        n_ho = holdout[col].str.contains(pat, regex=True).sum()
        flag = "" if n_tr > 0 else "   MISSING FROM TRAIN"
        print(f"{col} {t}: train {n_tr}, holdout {n_ho}{flag}")

groups on both sides: 0
Col4 Word575: train 128, holdout 14
Col4 Word868: train 29, holdout 7
Col4 Word905: train 20, holdout 4
Col6 Word576: train 120, holdout 10
Col6 Word571: train 186, holdout 22
Col1 Word19: train 182, holdout 21
Col1 Word85: train 182, holdout 17
Col1 Word101: train 43, holdout 3


Saving the holdout, and returning Category_5 to the modeling set

In [4]:
HOLD_DIR = ROOT / "data" / "holdout"
HOLD_DIR.mkdir(parents=True, exist_ok=True)

RAW_COLS = ["Col1", "Col2", "Col3", "Col4", "Col5", "Col6", "Col7"]

holdout[RAW_COLS].to_csv(HOLD_DIR / "holdout_features.csv", index=False)
holdout[["label"]].to_csv(HOLD_DIR / "holdout_labels.csv", index=False)
print("saved", len(holdout), "holdout rows")

# Category_5 rows rejoin the modeling set
model_df = pd.concat([train, cat5], ignore_index=True)
print("\nmodeling set:", len(model_df))
print(model_df["label"].value_counts())
print("groups:", model_df["group_key"].nunique())

assert not (set(model_df["group_key"]) & set(holdout["group_key"])), "holdout group leaked"
print("\nno overlap with holdout")

saved 590 holdout rows

modeling set: 5309
label
Category_1    4693
Category_2     568
Category_3      21
Category_4      14
Category_6      11
Category_5       2
Name: count, dtype: int64
groups: 4697

no overlap with holdout


Col2 structural features: shape masks, length, and the 4.80 family

In [5]:
new = ["col2_len", "col2_has_alpha", "col2_has_hyphen", "col2_is_480_family",
       "col2_shape", "col2_shape_coarse"]
print(df[new].head(10))

for c in ["col2_shape", "col2_shape_coarse"]:
    print(f"\n{c}: {df[c].nunique()} distinct")
    print(df[c].value_counts().head(8))

print("\n480 family rows:", df["col2_is_480_family"].sum())
print("has hyphen:", df["col2_has_hyphen"].sum(), "| has alpha:", df["col2_has_alpha"].sum())
print("\nlength stats:\n", df["col2_len"].describe())

for c in ["col2_has_alpha", "col2_has_hyphen", "col2_is_480_family"]:
    print(f"\nclass share by {c}:")
    print(pd.crosstab(df[c], df["label"], normalize="index").mul(100).round(2))

   col2_len  col2_has_alpha  col2_has_hyphen  col2_is_480_family  \
0         8           False            False               False   
1         5           False            False               False   
2         8            True            False                True   
3         5           False            False               False   
4         8            True            False                True   
5        16            True            False               False   
6         6            True            False               False   
7         5           False            False               False   
8         4           False            False               False   
9         8            True            False                True   

         col2_shape col2_shape_coarse  
0          99999999                 9  
1             99999                 9  
2          9.99A+99            9.9A+9  
3             99999                 9  
4          9.99A+99            9.9A+9  
5  AAAAAAAA

Custom Cross Validation (CV) splitter (extracted from src/splits.py), group-aware, with the Category_5 rows forced into every training fold. 

In [6]:
from src.splits import make_cv_splits

splits = make_cv_splits(model_df, n_splits=3, seed=42)

for i, (tr, va) in enumerate(splits, 1):
    tr_lab, va_lab = model_df.iloc[tr]["label"], model_df.iloc[va]["label"]
    g_tr, g_va = set(model_df.iloc[tr]["group_key"]), set(model_df.iloc[va]["group_key"])
    print(f"\nfold {i}: train {len(tr)}, val {len(va)} | group overlap: {len(g_tr & g_va)}")
    print(pd.DataFrame({"train": tr_lab.value_counts(),
                        "val": va_lab.value_counts()}).fillna(0).astype(int).T)


fold 1: train 3539, val 1770 | group overlap: 0
label  Category_1  Category_2  Category_3  Category_4  Category_5  Category_6
train        3129         378          14           9           2           7
val          1564         190           7           5           0           4

fold 2: train 3540, val 1769 | group overlap: 0
label  Category_1  Category_2  Category_3  Category_4  Category_5  Category_6
train        3128         379          14           9           2           8
val          1565         189           7           5           0           3

fold 3: train 3541, val 1768 | group overlap: 0
label  Category_1  Category_2  Category_3  Category_4  Category_5  Category_6
train        3129         379          14          10           2           7
val          1564         189           7           4           0           4


Building the feature matrix: 7 columns in, hundreds of features out

In [7]:
from src.preprocessing import build_features

ct = build_features()
Xt = ct.fit_transform(model_df)
print("shape:", Xt.shape)

names = ct.get_feature_names_out()
import collections
print(collections.Counter(n.split("__")[0] for n in names))

shape: (5309, 1481)
Counter({'col4_tfidf': 828, 'col1_tfidf': 378, 'col6_tfidf': 227, 'onehot': 34, 'date': 9, 'num': 5})


Chaining the classifier on, so raw columns go in and predictions come out

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import LabelEncoder

from src.preprocessing import build_pipeline, FEATURE_COLS

le = LabelEncoder().fit(model_df["label"])

pipe = build_pipeline(DummyClassifier(strategy="most_frequent"))
small = model_df.sample(300, random_state=42)
pipe.fit(small, le.transform(small["label"]))

raw_unseen = holdout[FEATURE_COLS].head(5)
print(le.inverse_transform(pipe.predict(raw_unseen)))
print("input columns:", list(raw_unseen.columns))

['Category_1' 'Category_1' 'Category_1' 'Category_1' 'Category_1']
input columns: ['Col1', 'Col2', 'Col3', 'Col4', 'Col5', 'Col6', 'Col7']


Baseline Model - Dummy Classifier (majority class every time)

In [9]:
from src.evaluate import cv_evaluate

results = {}

dummy = build_pipeline(DummyClassifier(strategy="most_frequent"))
res = cv_evaluate(dummy, model_df, le)
results["dummy"] = res

print("macro F1:", round(res["macro_f1"], 4))
print("balanced acc:", round(res["balanced_acc"], 4))
print("accuracy:", round(res["accuracy"], 4))
print("per fold:", [round(s, 4) for s in res["fold_macro_f1"]])
print("rows validated:", res["n_validated"], "of", len(model_df))
print("\n", res["report"])

macro F1: 0.1877
balanced acc: 0.2
accuracy: 0.8843
per fold: [0.1876, 0.1878, 0.1878]
rows validated: 5307 of 5309

               precision    recall  f1-score   support

  Category_1      0.884     1.000     0.939      4693
  Category_2      0.000     0.000     0.000       568
  Category_3      0.000     0.000     0.000        21
  Category_4      0.000     0.000     0.000        14
  Category_6      0.000     0.000     0.000        11

    accuracy                          0.884      5307
   macro avg      0.177     0.200     0.188      5307
weighted avg      0.782     0.884     0.830      5307



Model Ladder - Evaluating Models: Logistic Regression, Random Forest, Gradient Boosting

In [10]:
import time
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

candidates = {
    "logreg": (LogisticRegression(class_weight="balanced", max_iter=5000,
                                  C=1.0, random_state=42), False),
    "rf": (RandomForestClassifier(class_weight="balanced", n_estimators=300,
                                  n_jobs=-1, random_state=42), False),
    "hgb": (HistGradientBoostingClassifier(class_weight="balanced",
                                           random_state=42), True),
}

for name, (clf, dense) in candidates.items():
    t0 = time.time()
    res = cv_evaluate(build_pipeline(clf, dense=dense), model_df, le)
    results[name] = res
    print(f"{name:8s} macro F1 {res['macro_f1']:.4f} "
          f"| bal acc {res['balanced_acc']:.4f} "
          f"| acc {res['accuracy']:.4f} "
          f"| folds {[round(s, 3) for s in res['fold_macro_f1']]} "
          f"| {time.time()-t0:.0f}s")

summary = pd.DataFrame({
    k: {"macro_f1": v["macro_f1"], "balanced_acc": v["balanced_acc"],
        "accuracy": v["accuracy"], "f1_spread": max(v["fold_macro_f1"]) - min(v["fold_macro_f1"])}
    for k, v in results.items()
}).T.round(4)
summary

/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as

logreg   macro F1 0.6563 | bal acc 0.8458 | acc 0.9005 | folds [0.666, 0.608, 0.692] | 22s
rf       macro F1 0.7554 | bal acc 0.8018 | acc 0.9367 | folds [0.79, 0.677, 0.788] | 2s
hgb      macro F1 0.7022 | bal acc 0.7025 | acc 0.9412 | folds [0.696, 0.578, 0.791] | 72s


/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,macro_f1,balanced_acc,accuracy,f1_spread
dummy,0.1877,0.2000,0.8843,0.0001
logreg,0.6563,0.8458,0.9005,0.0836
rf,0.7554,0.8018,0.9367,0.1129
hgb,0.7022,0.7025,0.9412,0.2138


5 Class Weightings Compared on the winner model

In [11]:
counts = model_df["label"].value_counts()
n, k = len(model_df), len(counts)
enc = {c: le.transform([c])[0] for c in counts.index}

balanced = {enc[c]: n / (k * v) for c, v in counts.items()}
sqrt_bal = {i: np.sqrt(w) for i, w in balanced.items()}
capped20 = {i: min(w, 20.0) for i, w in balanced.items()}

print(pd.DataFrame({
    "count": {c: counts[c] for c in counts.index},
    "balanced": {c: round(balanced[enc[c]], 1) for c in counts.index},
    "sqrt": {c: round(sqrt_bal[enc[c]], 1) for c in counts.index},
    "capped20": {c: round(capped20[enc[c]], 1) for c in counts.index},
}))

weightings = {
    "rf_none": None,
    "rf_balanced": "balanced",
    "rf_bal_subsample": "balanced_subsample",
    "rf_sqrt": sqrt_bal,
    "rf_capped20": capped20,
}

for name, w in weightings.items():
    clf = RandomForestClassifier(class_weight=w, n_estimators=300,
                                 n_jobs=-1, random_state=42)
    res = cv_evaluate(build_pipeline(clf), model_df, le)
    results[name] = res
    print(f"{name:18s} macro F1 {res['macro_f1']:.4f} "
          f"| bal acc {res['balanced_acc']:.4f} "
          f"| folds {[round(s, 3) for s in res['fold_macro_f1']]}")

            count  balanced  sqrt  capped20
Category_1   4693       0.2   0.4       0.2
Category_2    568       1.6   1.2       1.6
Category_3     21      42.1   6.5      20.0
Category_4     14      63.2   7.9      20.0
Category_6     11      80.4   9.0      20.0
Category_5      2     442.4  21.0      20.0
rf_none            macro F1 0.5939 | bal acc 0.5910 | folds [0.576, 0.577, 0.629]
rf_balanced        macro F1 0.7554 | bal acc 0.8018 | folds [0.79, 0.677, 0.788]
rf_bal_subsample   macro F1 0.5815 | bal acc 0.5677 | folds [0.568, 0.546, 0.642]
rf_sqrt            macro F1 0.6157 | bal acc 0.6391 | folds [0.64, 0.599, 0.614]
rf_capped20        macro F1 0.7057 | bal acc 0.7163 | folds [0.761, 0.624, 0.707]


Two-Stage Model Design (extracted from src/two_stage.py): Category_1 against the rest, then the rest

In [12]:
from src.two_stage import TwoStageClassifier

major = le.transform(["Category_1"])[0]
rf = lambda: RandomForestClassifier(class_weight="balanced", n_estimators=300,
                                    n_jobs=-1, random_state=42)

two = TwoStageClassifier(
    stage_a=build_pipeline(rf()),
    stage_b=build_pipeline(rf()),
    majority_code=major,
)

res = cv_evaluate(two, model_df, le)
results["two_stage_rf"] = res
print(f"two_stage_rf macro F1 {res['macro_f1']:.4f} "
      f"| bal acc {res['balanced_acc']:.4f} "
      f"| folds {[round(s, 3) for s in res['fold_macro_f1']]}")
print("\n", res["report"])
print("\n", res["confusion"])

two_stage_rf macro F1 0.6509 | bal acc 0.7047 | folds [0.722, 0.6, 0.633]

               precision    recall  f1-score   support

  Category_1      0.978     0.968     0.973      4693
  Category_2      0.771     0.822     0.796       568
  Category_3      0.808     1.000     0.894        21
  Category_4      0.346     0.643     0.450        14
  Category_6      0.333     0.091     0.143        11

   micro avg      0.950     0.950     0.950      5307
   macro avg      0.647     0.705     0.651      5307
weighted avg      0.952     0.950     0.950      5307


             Category_1  Category_2  Category_3  Category_4  Category_6
Category_1        4541         139           0          10           2
Category_2          94         467           0           7           0
Category_3           0           0          21           0           0
Category_4           2           0           3           9           0
Category_6           8           0           2           0           1


/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Comparison between single stage model ladder vs two-stage model design per class

In [13]:
print("SINGLE STAGE\n", results["rf_balanced"]["report"])
print(results["rf_balanced"]["confusion"])

SINGLE STAGE
               precision    recall  f1-score   support

  Category_1      0.984     0.946     0.965      4693
  Category_2      0.668     0.875     0.758       568
  Category_3      0.808     1.000     0.894        21
  Category_4      0.450     0.643     0.529        14
  Category_6      0.750     0.545     0.632        11

    accuracy                          0.937      5307
   macro avg      0.732     0.802     0.755      5307
weighted avg      0.948     0.937     0.940      5307

            Category_1  Category_2  Category_3  Category_4  Category_6
Category_1        4438         246           0           7           2
Category_2          67         497           0           4           0
Category_3           0           0          21           0           0
Category_4           1           1           3           9           0
Category_6           3           0           2           0           6


Give stage A per-row weights from the original six-class labels

In [14]:
two_w = TwoStageClassifier(
    stage_a=build_pipeline(RandomForestClassifier(
        n_estimators=300, n_jobs=-1, random_state=42)),   # no class_weight now
    stage_b=build_pipeline(rf()),
    majority_code=major,
    a_weights=balanced,        # the dict from step 3
)

res = cv_evaluate(two_w, model_df, le)
results["two_stage_weighted"] = res
print(f"two_stage_weighted macro F1 {res['macro_f1']:.4f} "
      f"| bal acc {res['balanced_acc']:.4f} "
      f"| folds {[round(s, 3) for s in res['fold_macro_f1']]}")
print("\n", res["report"])

two_stage_weighted macro F1 0.6791 | bal acc 0.8365 | folds [0.689, 0.616, 0.727]

               precision    recall  f1-score   support

  Category_1      0.989     0.924     0.955      4693
  Category_2      0.613     0.908     0.732       568
  Category_3      0.808     1.000     0.894        21
  Category_4      0.303     0.714     0.426        14
  Category_6      0.280     0.636     0.389        11

   micro avg      0.921     0.921     0.921      5307
   macro avg      0.599     0.837     0.679      5307
weighted avg      0.945     0.921     0.929      5307



/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Randomized search over forest settings, min_df, and the date branch

In [15]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

from src.evaluate import macro_f1_scorer
from src.preprocessing import DateFeatures
from src.splits import make_cv_splits

base = build_pipeline(RandomForestClassifier(
    class_weight="balanced", n_jobs=1, random_state=42))

param_dist = {
    # pipeline choices, still unresolved
    "features__col1_tfidf__min_df": [1, 2, 3, 5],
    "features__col4_tfidf__min_df": [1, 2, 3, 5],
    "features__col6_tfidf__min_df": [1, 2, 3, 5],
    "features__date": [DateFeatures(), "drop"],
    # forest settings
    "clf__n_estimators": randint(200, 800),
    "clf__max_depth": [None, 10, 20, 30, 50],
    "clf__min_samples_leaf": randint(1, 5),
    "clf__min_samples_split": randint(2, 12),
    "clf__max_features": ["sqrt", "log2", 0.1, 0.3],
}

search = RandomizedSearchCV(
    base, param_dist, n_iter=40,
    scoring=macro_f1_scorer,
    cv=make_cv_splits(model_df, n_splits=3, seed=42),
    refit=True, n_jobs=-1, random_state=42, verbose=1,
)

y_all = le.transform(model_df["label"])
search.fit(model_df, y_all)

print("best macro F1:", round(search.best_score_, 4))
for k, v in sorted(search.best_params_.items()):
    print(f"  {k}: {v}")

Fitting 3 folds for each of 40 candidates, totalling 120 fits
best macro F1: 0.7588
  clf__max_depth: None
  clf__max_features: log2
  clf__min_samples_leaf: 1
  clf__min_samples_split: 2
  clf__n_estimators: 433
  features__col1_tfidf__min_df: 5
  features__col4_tfidf__min_df: 5
  features__col6_tfidf__min_df: 3
  features__date: DateFeatures()


Figure setup

In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGDIR = ROOT / "artifacts" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGDIR / f"{name}.png", dpi=120)
    plt.close(fig)
    print("saved", name)

Search results, and whether the date branch earns its place

In [17]:
cv = pd.DataFrame(search.cv_results_)
cv.to_csv(ROOT / "artifacts" / "cv_results.csv", index=False)
print("candidates:", len(cv), "| score range:",
      round(cv["mean_test_score"].min(), 4), "to", round(cv["mean_test_score"].max(), 4))

# does the date branch actually help
cv["has_dates"] = cv["param_features__date"].astype(str).ne("drop")
print("\ndate branch:")
print(cv.groupby("has_dates")["mean_test_score"].agg(["count", "mean", "max"]).round(4))

params = ["clf__n_estimators", "clf__max_depth", "clf__min_samples_leaf",
          "clf__min_samples_split", "clf__max_features",
          "features__col1_tfidf__min_df", "features__col4_tfidf__min_df",
          "features__col6_tfidf__min_df"]

y = cv["mean_test_score"].to_numpy(dtype=float)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, p in zip(axes.ravel(), params):
    vals = [str(v) for v in cv[f"param_{p}"]]
    levels = sorted(set(vals))
    if len(levels) > 10:                       # continuous, plot as numbers
        ax.scatter(cv[f"param_{p}"].astype(float), y, alpha=0.6)
    else:                                      # categorical, plot positions
        ax.scatter([levels.index(v) for v in vals], y, alpha=0.6)
        ax.set_xticks(range(len(levels)))
        ax.set_xticklabels(levels, rotation=45, fontsize=8)
    ax.axhline(search.best_score_, ls="--", c="r", lw=1)
    ax.set_title(p.split("__")[-1], fontsize=10)
    ax.set_ylabel("macro F1")
save(fig, "13_hyperparam_vs_score")

candidates: 40 | score range: 0.4414 to 0.7588

date branch:
           count    mean     max
has_dates                       
False         19  0.6257  0.7248
True          21  0.6227  0.7588
saved 13_hyperparam_vs_score


Score Distribution - To check whether the search plateaued or the winner got lucky

In [18]:
best = search.best_score_
near = cv[cv["mean_test_score"] >= best - 0.01]
print(f"best {best:.4f} | candidates within 0.01: {len(near)} of {len(cv)}")
print(f"candidates above the untuned 0.7554: {(cv['mean_test_score'] > 0.7554).sum()}")
print("\ntop 8:")
print(cv.nlargest(8, "mean_test_score")[
    ["mean_test_score", "std_test_score",
     "param_clf__max_features", "param_clf__n_estimators",
     "param_features__col4_tfidf__min_df"]].round(4).to_string())

best 0.7588 | candidates within 0.01: 1 of 40
candidates above the untuned 0.7554: 1

top 8:
    mean_test_score  std_test_score param_clf__max_features  param_clf__n_estimators  param_features__col4_tfidf__min_df
23           0.7588          0.0619                    log2                      433                                   5
20           0.7248          0.0701                    log2                      227                                   5
37           0.7052          0.0256                     0.1                      537                                   2
34           0.7047          0.0577                    sqrt                      208                                   3
13           0.7002          0.0210                     0.1                      689                                   2
21           0.6931          0.0496                     0.3                      547                                   3
18           0.6862          0.0747                     0.3 

Three configurations (untuned, search_winner, and evidence) re-scored across three seeds

In [19]:
configs = {
    "untuned": dict(clf=dict(n_estimators=300), md=(2, 2, 2), dates=True),
    "search_winner": dict(clf=dict(n_estimators=433, max_features="log2",
                                   min_samples_leaf=1, max_depth=None),
                          md=(5, 5, 3), dates=True),
    "evidence": dict(clf=dict(n_estimators=500, max_features="sqrt",
                              min_samples_leaf=1, max_depth=None,
                              min_samples_split=2),
                     md=(5, 5, 3), dates=False),
}

rows = []
for name, cfg in configs.items():
    for seed in [42, 7, 2024]:
        pipe = build_pipeline(
            RandomForestClassifier(class_weight="balanced", n_jobs=-1,
                                   random_state=42, **cfg["clf"]),
            use_dates=cfg["dates"])
        # per-column min_df
        for col, m in zip(["col1", "col4", "col6"], cfg["md"]):
            pipe.named_steps["features"].set_params(**{f"{col}_tfidf__min_df": m})
        r = cv_evaluate(pipe, model_df, le, seed=seed)
        rows.append({"config": name, "seed": seed, "macro_f1": r["macro_f1"]})
        print(f"{name:14s} seed {seed:5d}  {r['macro_f1']:.4f}")

comp = pd.DataFrame(rows).pivot(index="config", columns="seed", values="macro_f1")
comp["mean"] = comp.mean(axis=1)
comp["std"] = comp.std(axis=1)
comp.round(4)

untuned        seed    42  0.7554
untuned        seed     7  0.7284
untuned        seed  2024  0.7137
search_winner  seed    42  0.7601
search_winner  seed     7  0.7255
search_winner  seed  2024  0.7502


/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


evidence       seed    42  0.7380


/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


evidence       seed     7  0.7143
evidence       seed  2024  0.7717


/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


seed,7,42,2024,mean,std
config,,,,,
evidence,0.7143,0.7380,0.7717,0.7414,0.0236
search_winner,0.7255,0.7601,0.7502,0.7452,0.0145
untuned,0.7284,0.7554,0.7137,0.7325,0.0172


Isolating the date branch, changing nothing else

In [20]:
cfg = dict(n_estimators=433, max_features="log2",
           min_samples_leaf=1, max_depth=None)

rows = []
for dates in [True, False]:
    for seed in [42, 7, 2024]:
        pipe = build_pipeline(
            RandomForestClassifier(class_weight="balanced", n_jobs=-1,
                                   random_state=42, **cfg),
            use_dates=dates)
        for col, m in zip(["col1", "col4", "col6"], (5, 5, 3)):
            pipe.named_steps["features"].set_params(**{f"{col}_tfidf__min_df": m})
        r = cv_evaluate(pipe, model_df, le, seed=seed)
        rows.append({"dates": dates, "seed": seed, "macro_f1": r["macro_f1"]})

d = pd.DataFrame(rows).pivot(index="dates", columns="seed", values="macro_f1")
d["mean"] = d.mean(axis=1)
d.round(4)

/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


seed,7,42,2024,mean
dates,,,,
False,0.7139,0.7345,0.7723,0.7402
True,0.7255,0.7601,0.7502,0.7452


Final Configuration

In [21]:
final_clf = RandomForestClassifier(
    class_weight="balanced", n_estimators=433, max_features="log2",
    min_samples_leaf=1, max_depth=None, n_jobs=-1, random_state=42)
final_min_df = (5, 5, 3)     # col1, col4, col6
final_use_dates = False

Tuning per-class thresholds on out-of-fold predictions

In [22]:
from src.evaluate import cv_oof_proba
from sklearn.metrics import classification_report, f1_score

pipe = build_pipeline(final_clf, use_dates=final_use_dates)
for col, m in zip(["col1", "col4", "col6"], final_min_df):
    pipe.named_steps["features"].set_params(**{f"{col}_tfidf__min_df": m})

proba, y_all = cv_oof_proba(pipe, model_df, le)
mask = proba.sum(axis=1) > 0          # excludes the 2 Category_5 rows
P, y_true = proba[mask], y_all[mask]
seen = np.unique(y_true)

cat5 = le.transform(["Category_5"])[0]

def score(w):
    pred = (P * w).argmax(axis=1)
    return f1_score(y_true, pred, labels=seen, average="macro", zero_division=0)

w = np.ones(len(le.classes_))
print("start:", round(score(w), 4))

grid = np.round(np.exp(np.linspace(np.log(0.2), np.log(5), 25)), 3)
for rnd in range(4):
    for c in range(len(w)):
        if c == cat5:
            continue                   # 2 rows cannot justify a tuned weight
        best_v, best_s = w[c], score(w)
        for v in grid:
            w[c] = v
            s = score(w)
            if s > best_s:
                best_v, best_s = v, s
        w[c] = best_v
    print(f"round {rnd+1}: {score(w):.4f}")

print("\nweights:", dict(zip(le.classes_, w.round(3))))
print(classification_report(y_true, (P * w).argmax(axis=1), labels=seen,
                            target_names=le.classes_[seen], zero_division=0, digits=3))

start: 0.7345
round 1: 0.7545
round 2: 0.7601
round 3: 0.7601
round 4: 0.7601

weights: {'Category_1': np.float64(0.874), 'Category_2': np.float64(0.299), 'Category_3': np.float64(1.0), 'Category_4': np.float64(0.262), 'Category_5': np.float64(1.0), 'Category_6': np.float64(1.0)}
              precision    recall  f1-score   support

  Category_1      0.973     0.977     0.975      4693
  Category_2      0.817     0.778     0.797       568
  Category_3      0.808     1.000     0.894        21
  Category_4      0.533     0.571     0.552        14
  Category_6      0.538     0.636     0.583        11

    accuracy                          0.954      5307
   macro avg      0.734     0.793     0.760      5307
weighted avg      0.953     0.954     0.953      5307



Freezing those weights and applying them to unseen fold arrangements

In [23]:
w_frozen = w.copy()

for seed in [7, 2024]:
    p2, y2 = cv_oof_proba(pipe, model_df, le, seed=seed)
    m2 = p2.sum(axis=1) > 0
    P2, yt2 = p2[m2], y2[m2]
    s2 = np.unique(yt2)
    base = f1_score(yt2, P2.argmax(axis=1), labels=s2, average="macro", zero_division=0)
    tuned = f1_score(yt2, (P2 * w_frozen).argmax(axis=1), labels=s2,
                     average="macro", zero_division=0)
    print(f"seed {seed}: base {base:.4f} -> tuned {tuned:.4f}  ({tuned-base:+.4f})")

seed 7: base 0.7139 -> tuned 0.6972  (-0.0167)
seed 2024: base 0.7723 -> tuned 0.7833  (+0.0111)


In [24]:
print("VERDICT: weights do not generalize across seeds. "
      "Final model uses unweighted argmax. No thresholds saved.")

VERDICT: weights do not generalize across seeds. Final model uses unweighted argmax. No thresholds saved.


Category_5 leave-one-out, at n=2 a sanity check rather than a score

In [25]:
from sklearn.base import clone

final_clf = RandomForestClassifier(
    class_weight="balanced", n_estimators=433, max_features="log2",
    min_samples_leaf=1, max_depth=None, n_jobs=-1, random_state=42)

def make_final_pipe():
    p = build_pipeline(clone(final_clf), use_dates=False)
    for col, m in zip(["col1", "col4", "col6"], (5, 5, 3)):
        p.named_steps["features"].set_params(**{f"{col}_tfidf__min_df": m})
    return p

cat5_pos = np.flatnonzero(model_df["label"].values == "Category_5")
y_all = le.transform(model_df["label"])

print("leave-one-out on Category_5 (n=2, sanity check only)\n")
for i in cat5_pos:
    keep = np.setdiff1d(np.arange(len(model_df)), [i])
    m = make_final_pipe()
    m.fit(model_df.iloc[keep], y_all[keep])
    p = m.predict_proba(model_df.iloc[[i]])[0]
    pred = le.classes_[p.argmax()]
    c5 = p[le.transform(["Category_5"])[0]]
    print(f"row {i}: predicted {pred} | P(Category_5) = {c5:.4f}")
    print("   top 3:", [(le.classes_[j], round(p[j], 3)) for j in p.argsort()[::-1][:3]])

leave-one-out on Category_5 (n=2, sanity check only)

row 5307: predicted Category_1 | P(Category_5) = 0.1594
   top 3: [('Category_1', np.float64(0.476)), ('Category_2', np.float64(0.333)), ('Category_5', np.float64(0.159))]
row 5308: predicted Category_2 | P(Category_5) = 0.1570
   top 3: [('Category_2', np.float64(0.467)), ('Category_1', np.float64(0.372)), ('Category_5', np.float64(0.157))]


Observation - Category_5 ranked third of six at roughly 400 times its base rate

Refitting on the full modeling set and saving pipeline.joblib, metrics.json, and the confusion matrices


In [26]:
import json, joblib
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report, confusion_matrix

ART = ROOT / "artifacts"
ART.mkdir(exist_ok=True)

# CV numbers, recomputed on the exact final config across three seeds
cv_runs = {}
for seed in [42, 7, 2024]:
    r = cv_evaluate(make_final_pipe(), model_df, le, seed=seed)
    cv_runs[seed] = r
    print(f"seed {seed}: macro F1 {r['macro_f1']:.4f} | bal acc {r['balanced_acc']:.4f}")

cv_f1 = [r["macro_f1"] for r in cv_runs.values()]
print(f"\nCV macro F1: {np.mean(cv_f1):.4f} +/- {np.std(cv_f1):.4f}")

# refit on everything
final_pipe = make_final_pipe()
final_pipe.fit(model_df, y_all)

joblib.dump(final_pipe, ART / "pipeline.joblib", compress=3)
joblib.dump(le, ART / "label_encoder.joblib", compress=3)

r42 = cv_runs[42]
metrics = {
    "config": {
        "model": "RandomForestClassifier",
        "class_weight": "balanced", "n_estimators": 433,
        "max_features": "log2", "min_samples_leaf": 1, "max_depth": None,
        "min_df": {"Col1": 5, "Col4": 5, "Col6": 3},
        "date_branch": False, "thresholds": None,
    },
    "cv": {
        "n_splits": 3, "seeds": [42, 7, 2024],
        "macro_f1_mean": float(np.mean(cv_f1)),
        "macro_f1_std": float(np.std(cv_f1)),
        "macro_f1_by_seed": {str(k): float(v["macro_f1"]) for k, v in cv_runs.items()},
        "balanced_accuracy_seed42": float(r42["balanced_acc"]),
        "accuracy_seed42": float(r42["accuracy"]),
        "n_validated": r42["n_validated"],
    },
    "baselines": {
        "dummy_macro_f1_seed42": 0.1877,
        "dummy_accuracy_seed42": 0.8843,
        "untuned_rf_macro_f1_mean_3seeds": 0.7325,
        "untuned_rf_macro_f1_seed42": 0.7554,
    },
    "rejected": {
        "two_stage": 0.6509, "two_stage_weighted": 0.6791,
        "threshold_tuning": "did not generalize across seeds",
        "date_branch": "no measurable effect",
        "col4_missing_indicator": "91.5% vs 88.3% Category_1, not meaningful",
    },
    "notes": {
        "metric_scope": "macro F1 over the 5 classes present in validation; Category_5 never validated",
        "accuracy_ceiling": 0.9997,
        "train_rows": len(model_df),
    },
}
(ART / "metrics.json").write_text(json.dumps(metrics, indent=2))

r42["confusion"].to_csv(ART / "confusion_matrix_cv.csv")
(ART / "classification_report_cv.txt").write_text(r42["report"])
print("\nsaved: pipeline.joblib, label_encoder.joblib, metrics.json, "
      "confusion_matrix_cv.csv, classification_report_cv.txt")

seed 42: macro F1 0.7345 | bal acc 0.8064
seed 7: macro F1 0.7139 | bal acc 0.7798


/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


seed 2024: macro F1 0.7723 | bal acc 0.8285

CV macro F1: 0.7402 +/- 0.0242

saved: pipeline.joblib, label_encoder.joblib, metrics.json, confusion_matrix_cv.csv, classification_report_cv.txt


Every Model Comparison Table View

In [27]:
DROP = {"rf"}          # identical to rf_balanced; the ladder already used balanced weights

LABELS = {
    "dummy": "dummy (majority class)",
    "logreg": "logistic regression",
    "hgb": "hist gradient boosting",
    "rf_none": "rf, no class weight",
    "rf_balanced": "rf, balanced weights",
    "rf_bal_subsample": "rf, balanced_subsample",
    "rf_sqrt": "rf, sqrt-balanced",
    "rf_capped20": "rf, weights capped at 20",
    "two_stage_rf": "two-stage",
    "two_stage_weighted": "two-stage, weighted stage A",
}

rows = []
for name, r in results.items():
    if name in DROP:
        continue
    rows.append({
        "model": LABELS.get(name, name),
        "macro_f1": round(r["macro_f1"], 4),
        "balanced_accuracy": round(r["balanced_acc"], 4),
        "accuracy": round(r["accuracy"], 4),
        "fold_min": round(min(r["fold_macro_f1"]), 4),
        "fold_max": round(max(r["fold_macro_f1"]), 4),
        "note": "",
    })

r42 = cv_runs[42]
rows.append({
    "model": "FINAL (rf tuned, no dates)",
    "macro_f1": round(r42["macro_f1"], 4),
    "balanced_accuracy": round(r42["balanced_acc"], 4),
    "accuracy": round(r42["accuracy"], 4),
    "fold_min": round(min(r42["fold_macro_f1"]), 4),
    "fold_max": round(max(r42["fold_macro_f1"]), 4),
    "note": "seed 42 only; across seeds 42/7/2024 it averages 0.7402 vs 0.7325 untuned",
})

comp = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
comp.to_csv(ART / "model_comparison.csv", index=False)
comp

,model,macro_f1,balanced_accuracy,accuracy,fold_min,fold_max,note
4,"rf, balanced weights",0.7554,0.8018,0.9367,0.6775,0.7904,
10,"FINAL (rf tuned, no dates)",0.7345,0.8064,0.9378,0.6808,0.7741,seed 42 only; across seeds 42/7/2024 it averag...
7,"rf, weights capped at 20",0.7057,0.7163,0.9455,0.6241,0.7606,
2,hist gradient boosting,0.7022,0.7025,0.9412,0.5776,0.7914,
9,"two-stage, weighted stage A",0.6791,0.8365,0.9210,0.6163,0.7270,
1,logistic regression,0.6563,0.8458,0.9005,0.6084,0.6921,
8,two-stage,0.6509,0.7047,0.9495,0.6005,0.7222,
6,"rf, sqrt-balanced",0.6157,0.6391,0.9520,0.5987,0.6404,
3,"rf, no class weight",0.5939,0.5910,0.9489,0.5761,0.6291,
5,"rf, balanced_subsample",0.5815,0.5677,0.9478,0.5459,0.6425,


Per-class results across three seeds, where the single-seed reading reversed

In [28]:
def per_class_across_seeds(make_pipe, name):
    rows = []
    for seed in [42, 7, 2024]:
        r = cv_evaluate(make_pipe(), model_df, le, seed=seed)
        cm = r["confusion"]
        for cls in cm.index:
            tp = cm.loc[cls, cls]
            f1 = 2 * tp / (cm.loc[cls].sum() + cm[cls].sum())
            rows.append({"config": name, "seed": seed, "class": cls, "f1": f1})
    return rows

def make_untuned_pipe():
    return build_pipeline(RandomForestClassifier(
        class_weight="balanced", n_estimators=300, n_jobs=-1, random_state=42),
        use_dates=True)

rows = per_class_across_seeds(make_final_pipe, "final")
rows += per_class_across_seeds(make_untuned_pipe, "untuned")

pc = pd.DataFrame(rows).pivot_table(index="class", columns="config",
                                    values="f1", aggfunc=["mean", "std"])
pc.round(3)

/Users/sejalagarwal/Desktop/classification-problem/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


mean            std        
config      final untuned  final untuned
class                                   
Category_1  0.965   0.965  0.001   0.001
Category_2  0.765   0.767  0.004   0.008
Category_3  0.835   0.874  0.115   0.047
Category_4  0.538   0.488  0.045   0.059
Category_6  0.599   0.568  0.048   0.066

Scoring the holdout, once

In [29]:
from src.preprocessing import load_and_clean, clean_frame

hf = pd.read_csv(ROOT / "data/holdout/holdout_features.csv", parse_dates=["Col5"])
hl = pd.read_csv(ROOT / "data/holdout/holdout_labels.csv")

hf = clean_frame(hf)
y_hold = le.transform(hl["label"])
pred = final_pipe.predict(hf)

seen = np.unique(y_hold)
hold = {
    "macro_f1": float(f1_score(y_hold, pred, labels=seen, average="macro", zero_division=0)),
    "balanced_accuracy": float(balanced_accuracy_score(y_hold, pred)),
    "accuracy": float((y_hold == pred).mean()),
    "n": len(hf),
}
print(hold)
print("\n", classification_report(y_hold, pred, labels=seen,
                                  target_names=le.classes_[seen], zero_division=0, digits=3))
cm = pd.DataFrame(confusion_matrix(y_hold, pred, labels=seen),
                  index=le.classes_[seen], columns=le.classes_[seen])
print("\n", cm)

metrics["holdout"] = hold
(ART / "metrics.json").write_text(json.dumps(metrics, indent=2))
cm.to_csv(ART / "confusion_matrix_holdout.csv")

{'macro_f1': 0.900320572048367, 'balanced_accuracy': 0.9599343185550083, 'accuracy': 0.9338983050847458, 'n': 590}

               precision    recall  f1-score   support

  Category_1      0.982     0.943     0.962       522
  Category_2      0.651     0.857     0.740        63
  Category_3      1.000     1.000     1.000         2
  Category_4      0.667     1.000     0.800         2
  Category_6      1.000     1.000     1.000         1

    accuracy                          0.934       590
   macro avg      0.860     0.960     0.900       590
weighted avg      0.946     0.934     0.938       590


             Category_1  Category_2  Category_3  Category_4  Category_6
Category_1         492          29           0           1           0
Category_2           9          54           0           0           0
Category_3           0           0           2           0           0
Category_4           0           0           0           2           0
Category_6           0           0   

Saving per-token class counts from the modeling set, for the explainer

In [30]:
import json

# per-token class counts from the MODELING set only, so explanations
# never quote statistics that came from holdout rows
token_stats = {}
for col in ["Col1", "Col4", "Col6"]:
    d = {}
    for toks, lab in zip(model_df[col].fillna("").str.split(), model_df["label"]):
        for t in set(toks):
            d.setdefault(t, {}).setdefault(lab, 0)
            d[t][lab] += 1
    token_stats[col] = d

payload = {
    "class_counts": model_df["label"].value_counts().to_dict(),
    "n_rows": len(model_df),
    "tokens": token_stats,
}
(ART / "token_stats.json").write_text(json.dumps(payload))
print("tokens saved:", {c: len(v) for c, v in token_stats.items()})

tokens saved: {'Col1': 538, 'Col4': 1489, 'Col6': 270}


Per-row attribution: ablation, after SHAP failed its additivity check

In [31]:
from src.explain import Contributions, build_evidence, load_token_stats, load_metrics

contrib = Contributions(final_pipe)          # ablation
stats, mets = load_token_stats(), load_metrics()

row = holdout[FEATURE_COLS].iloc[[5]]
ev = build_evidence(row, final_pipe, le, contrib, stats, mets)

print(ev["prediction"], ev["agreement"], "| method:", ev["attribution_method"])
print("runner-up:", ev["runner_up"], ev["runner_up_probability"])
print("\ntoward:")
for f in ev["pushing_toward"][:6]:
    print(f"  {f['readable']:42s} {f['contribution']:+.4f}  -> {f.get('probability without it')}")
print("\nagainst:")
for f in ev["pushing_against"][:4]:
    print(f"  {f['readable']:42s} {f['contribution']:+.4f}")

Category_2 0.7298 | method: ablation
runner-up: Category_1 0.2702

toward:
  col2 shape coarse = 9                      +0.1039  -> 0.6259
  Col7 = NoDoc                               +0.0831  -> 0.6467
  Col6 contains Word563                      +0.0785  -> 0.6513
  col6 first token = Word563                 +0.0762  -> 0.6536
  Col1 contains Word58                       +0.0693  -> 0.6605
  col2_len                                   +0.0670  -> 0.6628

against:
  Col4 contains Word824                      -0.0855
  Col4 contains Word789                      -0.0693
  Col4 contains Word816                      -0.0162


The Agentic Prediction Evaluator, and the offline fallback

In [32]:
from src.agent import explain

text, trace, used = explain(ev, provider="offline")
print(f"--- {used} ---\n{text}\n")

text, trace, used = explain(ev, provider="groq")
print(f"--- {used} ---\n{text}")
for t in trace:
    print(f"\ntool: {t['tool']}({t['input']})\n  -> {t['output']}")

--- offline ---
The model predicts Category_2, with 73.0% of the trees agreeing. Its second choice was Category_1 at 27.0%. The strongest support came from col2 shape coarse = 9 (removing it drops the probability to 62.6%); Col7 = NoDoc (removing it drops the probability to 64.7%); Col6 contains Word563 (removing it drops the probability to 65.1%). In training, Col1 containing Word58 appeared in 60 rows and was Category_2 52% of the time, 4.8 times that class's base rate. Working against it: Col4 contains Word824 (-0.086). (Generated without a language model. Set GROQ_API_KEY for a written explanation.)

--- groq ---
This transaction was predicted as Category_2 with 73.0% of trees agreeing, and the runner-up was Category_1 at 27.0%. The strongest driver for this prediction was the feature "col2 shape coarse = 9", which would drop the predicted probability to 62.59% if removed. Another key feature was "Col7 = NoDoc", which would drop the predicted probability to 64.67% if removed. The f

To confirm nothing changed in the notebook, on compressing pipeline.joblib in Retting cell

In [33]:
import joblib
reloaded = joblib.load(ART / "pipeline.joblib")
same = (reloaded.predict(holdout[FEATURE_COLS]) == final_pipe.predict(holdout[FEATURE_COLS])).all()
print("identical predictions:", same)

identical predictions: True
